In [2]:
# Predict the price of the Uber ride from a given pickup point to the agreed drop-off location.
# Perform following tasks:
# 1. Pre-process the dataset.
# 2. Identify outliers.
# 3. Check the correlation.
# 4. Implement linear regression and random forest regression models.
# 5. Evaluate the models and compare their respective scores like R2, RMSE, etc.
# Dataset link: https://www.kaggle.com/datasets/yasserh/uber-fares-dataset

In [4]:
import pandas as pd

In [12]:
df = pd.read_csv("uber.csv")

In [13]:
df.describe()

,Unnamed: 0,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,2.000000e+05,200000.000000,200000.000000,200000.000000,199999.000000,199999.000000,200000.000000
mean,2.771250e+07,11.359955,-72.527638,39.935885,-72.525292,39.923890,1.684535
std,1.601382e+07,9.901776,11.437787,7.720539,13.117408,6.794829,1.385997
min,1.000000e+00,-52.000000,-1340.648410,-74.015515,-3356.666300,-881.985513,0.000000
25%,1.382535e+07,6.000000,-73.992065,40.734796,-73.991407,40.733823,1.000000
50%,2.774550e+07,8.500000,-73.981823,40.752592,-73.980093,40.753042,1.000000
75%,4.155530e+07,12.500000,-73.967154,40.767158,-73.963658,40.768001,2.000000
max,5.542357e+07,499.000000,57.418457,1644.421482,1153.572603,872.697628,208.000000


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Unnamed: 0         200000 non-null  int64  
 1   key                200000 non-null  str    
 2   fare_amount        200000 non-null  float64
 3   pickup_datetime    200000 non-null  str    
 4   pickup_longitude   200000 non-null  float64
 5   pickup_latitude    200000 non-null  float64
 6   dropoff_longitude  199999 non-null  float64
 7   dropoff_latitude   199999 non-null  float64
 8   passenger_count    200000 non-null  int64  
dtypes: float64(5), int64(2), str(2)
memory usage: 13.7 MB


In [15]:
df.head(5)

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [16]:
# we will drop unnamed as it isnt significant and key as its redundant due to pickup_datetime
df = df.drop(["Unnamed: 0","key"],axis=1)
df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [18]:
# the dropoff loc had one null value so we removed the entire row as the dataset is far greater, and it wont affect the computation much
df = df.dropna()
df.info()

<class 'pandas.DataFrame'>
Index: 199999 entries, 0 to 199999
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   fare_amount        199999 non-null  float64
 1   pickup_datetime    199999 non-null  str    
 2   pickup_longitude   199999 non-null  float64
 3   pickup_latitude    199999 non-null  float64
 4   dropoff_longitude  199999 non-null  float64
 5   dropoff_latitude   199999 non-null  float64
 6   passenger_count    199999 non-null  int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 12.2 MB


In [19]:
# Using .sum() on the boolean condition (True evaluates to 1)
valid_count = (
    (df["pickup_latitude"].between(-90, 90)) & 
    (df["pickup_longitude"].between(-180, 180))
).sum()

print(f"Valid coordinates: {valid_count}")

Valid coordinates: 199989


In [20]:
# Using .sum() on the boolean condition (True evaluates to 1)
valid_count = (
    (df["dropoff_latitude"].between(-90, 90)) & 
    (df["dropoff_longitude"].between(-180, 180))
).sum()

print(f"Valid coordinates: {valid_count}")

Valid coordinates: 199995


In [21]:
# Using .sum() on the boolean condition (True evaluates to 1)
valid_count = (
    (df["dropoff_latitude"].between(-90, 90)) & 
    (df["dropoff_longitude"].between(-180, 180)) &
     (df["pickup_latitude"].between(-90, 90)) & 
    (df["pickup_longitude"].between(-180, 180))
).sum()

print(f"Valid coordinates: {valid_count}")

Valid coordinates: 199987


In [26]:
# Count rows where ANY coordinate is out of bounds
invalid_count = (
    ~df["dropoff_latitude"].between(-90, 90) |
    ~df["dropoff_longitude"].between(-180, 180) |
    ~df["pickup_latitude"].between(-90, 90) |
    ~df["pickup_longitude"].between(-180, 180)
).sum()

print(f"Rows with at least one invalid coordinate: {invalid_count}")

Rows with at least one invalid coordinate: 12


In [29]:
#since only 12 values were out of range we remove them instead of using normalization
# 1. Create the mask for invalid rows
invalid_mask = (
    ~df["dropoff_latitude"].between(-90, 90) |
    ~df["dropoff_longitude"].between(-180, 180) |
    ~df["pickup_latitude"].between(-90, 90) |
    ~df["pickup_longitude"].between(-180, 180)
)

# 2. Keep only the rows that are NOT invalid (~ flips True/False)
df = df[~invalid_mask]

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5
...,...,...,...,...,...,...,...
199995,3.0,2012-10-28 10:49:00 UTC,-73.987042,40.739367,-73.986525,40.740297,1
199996,7.5,2014-03-14 01:09:00 UTC,-73.984722,40.736837,-74.006672,40.739620,1
199997,30.9,2009-06-29 00:42:00 UTC,-73.986017,40.756487,-73.858957,40.692588,2
199998,14.5,2015-05-20 14:56:25 UTC,-73.997124,40.725452,-73.983215,40.695415,1


In [28]:
# Using .sum() on the boolean condition (True evaluates to 1)
valid_count = (
    (df["pickup_latitude"].between(-90, 90)) & 
    (df["pickup_longitude"].between(-180, 180))
).sum()

print(f"Valid coordinates: {valid_count}")

Valid coordinates: 199987


In [34]:
# 1. Convert the string column to datetime format
# (errors='coerce' safely turns bad/malformed dates into NaT instead of crashing)
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')

# 2. Extract the year, count occurrences, and sort by year chronologically
year_counts = df['pickup_datetime'].dt.year.value_counts().sort_index()

print(year_counts)

pickup_datetime
2009    30535
2010    30193
2011    31941
2012    32390
2013    31194
2014    29968
2015    13766
Name: count, dtype: int64


In [35]:
# Convert to datetime first (if you haven't already)
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')

# Extract features
df['year'] = df['pickup_datetime'].dt.year
df['month'] = df['pickup_datetime'].dt.month
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek  # 0=Monday, 6=Sunday
df['hour'] = df['pickup_datetime'].dt.hour

# Drop the original string column since Random Forest can't read strings
df = df.drop(columns=['pickup_datetime'])

In [36]:
df.sample(5)

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day_of_week,hour
147906,5.3,-73.986897,40.750863,-73.982038,40.744223,5,2010,8,5,13
61139,6.1,-73.989926,40.743936,-73.979044,40.752888,1,2011,6,1,18
49250,6.5,-73.955726,40.767855,-73.956253,40.781352,1,2012,1,2,13
172640,10.9,-73.952170,40.784072,-73.990122,40.761852,1,2012,8,5,19
117785,6.1,-73.970902,40.783342,-73.955665,40.769253,1,2009,11,6,13


In [37]:
import numpy as np

def calculate_haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # The math
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Radius of the Earth in kilometers is 6371 (use 3956 for miles)
    distance_km = 6371 * c
    return distance_km

# Apply the function to create a new feature
df['trip_distance_km'] = calculate_haversine(
    df['pickup_latitude'], 
    df['pickup_longitude'], 
    df['dropoff_latitude'], 
    df['dropoff_longitude']
)

In [38]:
df.head()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day_of_week,hour,trip_distance_km
0,7.5,-73.999817,40.738354,-73.999512,40.723217,1,2015,5,3,19,1.683323
1,7.7,-73.994355,40.728225,-73.994710,40.750325,1,2009,7,4,20,2.457590
2,12.9,-74.005043,40.740770,-73.962565,40.772647,1,2009,8,0,21,5.036377
3,5.3,-73.976124,40.790844,-73.965316,40.803349,3,2009,6,4,8,1.661683
4,16.0,-73.925023,40.744085,-73.973082,40.761247,5,2014,8,3,17,4.475450


In [39]:
df.info()

<class 'pandas.DataFrame'>
Index: 199987 entries, 0 to 199999
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   fare_amount        199987 non-null  float64
 1   pickup_longitude   199987 non-null  float64
 2   pickup_latitude    199987 non-null  float64
 3   dropoff_longitude  199987 non-null  float64
 4   dropoff_latitude   199987 non-null  float64
 5   passenger_count    199987 non-null  int64  
 6   year               199987 non-null  int32  
 7   month              199987 non-null  int32  
 8   day_of_week        199987 non-null  int32  
 9   hour               199987 non-null  int32  
 10  trip_distance_km   199987 non-null  float64
dtypes: float64(6), int32(4), int64(1)
memory usage: 15.3 MB


In [40]:
df.describe()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day_of_week,hour,trip_distance_km
count,199987.000000,199987.000000,199987.000000,199987.000000,199987.000000,199987.000000,199987.000000,199987.000000,199987.000000,199987.000000,199987.000000
mean,11.359849,-72.501786,39.917937,-72.511608,39.922031,1.684544,2011.742463,6.281783,3.048383,13.491562,20.560330
std,9.901868,10.449955,6.130412,10.412192,6.117669,1.385999,1.856438,3.438957,1.946960,6.515458,379.620738
min,-52.000000,-93.824668,-74.015515,-75.458979,-74.015750,0.000000,2009.000000,1.000000,0.000000,0.000000,0.000000
25%,6.000000,-73.992064,40.734793,-73.991407,40.733823,1.000000,2010.000000,3.000000,1.000000,9.000000,1.215167
50%,8.500000,-73.981822,40.752592,-73.980092,40.753042,1.000000,2012.000000,6.000000,3.000000,14.000000,2.120825
75%,12.500000,-73.967154,40.767157,-73.963658,40.768000,2.000000,2013.000000,9.000000,5.000000,19.000000,3.874779
max,499.000000,40.808425,48.018760,40.831932,45.031598,208.000000,2015.000000,12.000000,6.000000,23.000000,8782.898606


In [41]:
# 1. Keep valid fares (NYC base fare is ~$2.50)
df = df[df['fare_amount'] >= 2.5]

# 2. Keep valid passenger counts (1 to 6 people)
df = df[df['passenger_count'].between(1, 6)]

# 3. Restrict to NYC bounding box and reasonable distances (< 100 km)
df = df[
    (df['pickup_latitude'].between(40.5, 41.2)) &
    (df['pickup_longitude'].between(-74.3, -73.7)) &
    (df['dropoff_latitude'].between(40.5, 41.2)) &
    (df['dropoff_longitude'].between(-74.3, -73.7)) &
    (df['trip_distance_km'] > 0) & 
    (df['trip_distance_km'] < 100)
]

In [42]:
df.info()

<class 'pandas.DataFrame'>
Index: 192927 entries, 0 to 199999
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   fare_amount        192927 non-null  float64
 1   pickup_longitude   192927 non-null  float64
 2   pickup_latitude    192927 non-null  float64
 3   dropoff_longitude  192927 non-null  float64
 4   dropoff_latitude   192927 non-null  float64
 5   passenger_count    192927 non-null  int64  
 6   year               192927 non-null  int32  
 7   month              192927 non-null  int32  
 8   day_of_week        192927 non-null  int32  
 9   hour               192927 non-null  int32  
 10  trip_distance_km   192927 non-null  float64
dtypes: float64(6), int32(4), int64(1)
memory usage: 14.7 MB


In [43]:
df.describe()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day_of_week,hour,trip_distance_km
count,192927.000000,192927.000000,192927.000000,192927.000000,192927.000000,192927.000000,192927.000000,192927.000000,192927.000000,192927.000000,192927.000000
mean,11.325512,-73.975638,40.750860,-73.974703,40.751168,1.689618,2011.746293,6.283885,3.049770,13.488397,3.346723
std,9.575041,0.034317,0.026905,0.033941,0.030851,1.305217,1.863174,3.440955,1.946665,6.516761,3.579123
min,2.500000,-74.299012,40.508697,-74.299833,40.511859,1.000000,2009.000000,1.000000,0.000000,0.000000,0.000084
25%,6.000000,-73.992287,40.736497,-73.991606,40.735380,1.000000,2010.000000,3.000000,1.000000,9.000000,1.280599
50%,8.500000,-73.982140,40.753308,-73.980590,40.753761,1.000000,2012.000000,6.000000,3.000000,14.000000,2.180240
75%,12.500000,-73.968523,40.767539,-73.965619,40.768326,2.000000,2013.000000,9.000000,5.000000,19.000000,3.936063
max,499.000000,-73.702735,41.164293,-73.700650,41.161225,6.000000,2015.000000,12.000000,6.000000,23.000000,53.066101


In [44]:
# 1. Drop the raw coordinates
df = df.drop(columns=[
    'pickup_longitude', 'pickup_latitude', 
    'dropoff_longitude', 'dropoff_latitude'
])

# 2. Split into Features (X) and Target (y)
X = df.drop(columns=['fare_amount'])
y = df['fare_amount']

In [47]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Split data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train & Evaluate Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)



,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [64]:
# Added verbose=2 to show training progress
rf = RandomForestRegressor(n_estimators=80, max_depth=9, random_state=42, n_jobs=-1, verbose=1)

rf.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done  80 out of  80 | elapsed:    5.2s finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",80
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",9
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at

In [49]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

lr_preds = lr.predict(X_test)

# Calculate metrics
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_r2 = r2_score(y_test, lr_preds)

# Print results
print(f"Linear Regression RMSE: {lr_rmse:.2f}")
print(f"Linear Regression MAE:  {lr_mae:.2f}")
print(f"Linear Regression R²:   {lr_r2:.4f}")

Linear Regression RMSE: 4.45
Linear Regression MAE:  2.25
Linear Regression R²:   0.7774


In [50]:
rf_preds = rf.predict(X_test)

# Calculate metrics
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)

# Print results
print(f"Random Forest RMSE: {rf_rmse:.2f}")
print(f"Random Forest MAE:  {rf_mae:.2f}")
print(f"Random Forest R²:   {rf_r2:.4f}")

Random Forest RMSE: 4.21
Random Forest MAE:  2.04
Random Forest R²:   0.8013


[Parallel(n_jobs=14)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done  13 tasks      | elapsed:    0.0s
[Parallel(n_jobs=14)]: Done 100 out of 100 | elapsed:    0.0s finished


In [65]:
rf_preds = rf.predict(X_test)

# Calculate metrics
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)

# Print results
print(f"Random Forest RMSE: {rf_rmse:.2f}")
print(f"Random Forest MAE:  {rf_mae:.2f}")
print(f"Random Forest R²:   {rf_r2:.4f}")

[Parallel(n_jobs=14)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done  22 tasks      | elapsed:    0.0s
[Parallel(n_jobs=14)]: Done  80 out of  80 | elapsed:    0.1s finished


Random Forest RMSE: 4.19
Random Forest MAE:  2.04
Random Forest R²:   0.8032
